# Wskaźnik wykluczenia heksagonu — MLP (SiLU / GELU) — Google Colab

Sieć neuronowa (PyTorch MLP) przewidująca `etykieta_target` (0–1) dla heksagonów H3.
Dane pobierane są z **prywatnego repo GitHub przez token** (nic nie trzeba wgrywać ręcznie).

**Pliki z repo:** `dane/wskaznik_wykluczenia/h3_merged_clean.csv` (+ pomocniczo `compute_target.py`).
**Wymagania:** Colab (CPU wystarczy; GPU opcjonalnie), token GitHub z prawem `repo` (klasyczny) lub `contents:read` (fine-grained).
**Output:** `lista_heksagonow_ranking.csv` — heksagony posortowane malejąco po predykcji.

In [ ]:
# ===== KONFIGURACJA =====
REPO = "Hackathon-42Warsaw-The-Bills/3_TheBills_OptymalnaWarszawa"  # wlasciciel/nazwa
BRANCH = "wskaznik_wykluczenia_klastra"  # galaz z danymi (main NIE ma tych plikow)
FALLBACK_BRANCHES = ["main"]  # probowane po kolei gdy BRANCH zwroci 404
FILES = [
    "dane/wskaznik_wykluczenia/h3_merged_clean.csv",
    "dane/wskaznik_wykluczenia/compute_target.py",
    "dane/kontury_400m/warszawa_h3_400m_pelna.geojson",  # geometria do heatmapy
]  # pliki pobiera  ne z repo
LOCAL_DIR = "/content/dane_repo"  # gdzie zapisac pobrane pliki

# Parametry etykiety (musza zgadzac sie z compute_target.py)
ALPHA, W_TRAM, W_BUS, W_METRO, W_KOLEJ = 0.3, 1.5, 1.0, 3.0, 2.5

# Parametry sieci
HIDDEN = [32, 16]
HIDDEN_ACT = "silu"   # 'silu' | 'gelu' | 'relu'
OUTPUT_ACT = "silu"   # 'silu' | 'gelu' | 'none'  (patrz wyjasnienie w komorce modelu)
DROPOUT = 0.3
LR, WEIGHT_DECAY, BATCH, EPOCHS, PATIENCE, SEED = 1e-3, 1e-4, 64, 500, 40, 42

In [ ]:
# ===== TOKEN GITHUB (nie wpisuj na stale do notebooka) =====
import getpass, os
TOKEN = None
try:
    from google.colab import userdata  # Colab: zakladka Secrets (klucz: GITHUB_TOKEN_HACKATHON)
    TOKEN = userdata.get('GITHUB_TOKEN_HACKATHON')
    if TOKEN: print('token: Colab Secrets')
except Exception:
    pass
if not TOKEN:
    TOKEN = os.environ.get("GITHUB_TOKEN_HACKATHON")
    if TOKEN: print('token: zmienna srodowiskowa')
if not TOKEN:
    TOKEN = getpass.getpass("GitHub token (niewidoczny): ")
assert TOKEN and TOKEN.strip(), "Podaj token GitHub"
os.environ["GITHUB_TOKEN_HACKATHON"] = TOKEN  # zeby nie pytac ponownie po resecie UI
# szybki test uprawnien tokenu (contents:read do repo)
import urllib.request as _u
try:
    _r = _u.Request(f"https://api.github.com/repos/{REPO}/contents/", headers={"Authorization": f"Bearer {TOKEN}"})
    print('uprawnienia tokenu: OK, status', _u.urlopen(_r).status)
except Exception as e:
    print('UWAGA: test tokenu nie przeszedl:', e)

In [ ]:
# ===== POBRANIE PLIKOW Z REPO PRZEZ TOKEN =====
import os, urllib.request
os.makedirs(LOCAL_DIR, exist_ok=True)

def fetch_repo_file(repo_path, branch=BRANCH):
    last_err = None
    for br in [branch] + [b for b in FALLBACK_BRANCHES if b != branch]:
        url = f"https://raw.githubusercontent.com/{REPO}/{br}/{repo_path}"
        req = urllib.request.Request(url, headers={
            "Authorization": f"Bearer {TOKEN}",
            "Accept": "application/vnd.github.raw",
        })
        local = os.path.join(LOCAL_DIR, os.path.basename(repo_path))
        try:
            with urllib.request.urlopen(req) as r, open(local, "wb") as f:
                f.write(r.read())
            print(f"OK: {repo_path}@{br} -> {local} ({os.path.getsize(local)} B)")
            return local
        except Exception as e:
            last_err = f"[{br}] {e}"
    raise RuntimeError(f"Nie pobrano {repo_path}: {last_err}")

LOCAL = {p: fetch_repo_file(p) for p in FILES}
CSV_PATH = LOCAL["dane/wskaznik_wykluczenia/h3_merged_clean.csv"]
GEO_PATH = LOCAL["dane/kontury_400m/warszawa_h3_400m_pelna.geojson"]  # do heatmapy

In [ ]:
# ===== ZALEZNOSCI / IMPORTY =====
# Colab ma zwykle torch/sklearn/pandas; dociągnij tylko brakujące
import importlib, subprocess, sys
for pkg, mod in [("torch", "torch"), ("scikit-learn", "sklearn"), ("pandas", "pandas"), ("scipy", "scipy"), ("matplotlib", "matplotlib"), ("folium", "folium")]:
    try:
        importlib.import_module(mod)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", pkg])

import math, random
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import spearmanr
%matplotlib inline
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

In [ ]:
# ===== WCZYTANIE DANYCH (+ awaryjne policzenie etykiety jak w compute_target.py) =====
df = pd.read_csv(CSV_PATH)
print(df.shape, list(df.columns))
assert "etykieta_target" in df.columns, "Brak etykiety — uruchom najpierw compute_target.py w repo"

NUM = ["gestosc_os_km2", "pracownicy_szac", "przystanki_tramwajowe",
       "przystanki_autobusowe", "czy_metro", "czy_kolej", "wspolczynnik przyrostu", "lon", "lat"]
df[NUM] = df[NUM].apply(pd.to_numeric, errors="coerce").fillna(0.0)
print(df["etykieta_target"].describe().to_string())
print("top5 wg etykiety:", df.nlargest(5, "etykieta_target")["h3"].tolist())

In [ ]:
# ===== CECHY: log1p + one-hot dzielnicy + standaryzacja =====
USE_DZIELNICA = True
feat = df.copy()
feat["log_gest"] = np.log1p(feat["gestosc_os_km2"].clip(lower=0))
feat["log_prac"] = np.log1p(feat["pracownicy_szac"].clip(lower=0))
feat["ztm_w"] = W_TRAM * feat["przystanki_tramwajowe"] + W_BUS * feat["przystanki_autobusowe"] + W_METRO * (feat["czy_metro"] >= 0.5).astype(float) + W_KOLEJ * (feat["czy_kolej"] >= 0.5).astype(float)
feat["log_ztm"] = np.log1p(feat["ztm_w"].clip(lower=0))
base = ["log_gest", "log_prac", "log_ztm", "wspolczynnik przyrostu", "lon", "lat"]
X_num = feat[base].to_numpy(float)
if USE_DZIELNICA:
    dummies = pd.get_dummies(feat["dzielnica"].fillna("UNK"), prefix="dz").to_numpy(float)
    X = np.hstack([X_num, dummies])
    feat_names = base + list(pd.get_dummies(feat["dzielnica"].fillna("UNK"), prefix="dz").columns)
else:
    X, feat_names = X_num, base
y = feat["etykieta_target"].to_numpy(float)
print("cechy:", X.shape, "|", len(feat_names))

In [ ]:
# ===== PODZIAL PRZESTRZENNY (po dzielnicach, nie losowo) =====
groups = feat["dzielnica"].fillna("UNK").to_numpy()
gss = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=SEED)
tr, te = next(gss.split(X, y, groups))
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.176, random_state=SEED)  # ~15% calosci na walidacje
tr2, va = next(gss2.split(X[tr], y[tr], groups[tr]))
tr_idx, va_idx, te_idx = tr[tr2], tr[va], te
sc = StandardScaler().fit(X[tr_idx])
Xs = sc.transform(X[tr_idx]), sc.transform(X[va_idx]), sc.transform(X[te_idx])
print(f"train/val/test: {len(tr_idx)}/{len(va_idx)}/{len(te_idx)}")
print("dzielnice testowe:", sorted(set(groups[te_idx])))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

In [ ]:
# ===== MODEL: MLP z SiLU/GELU =====
# Ukryte: SiLU albo GELU (wybor HIDDEN_ACT). SiLU(x)=x*sigmoid(x) — gladka, nie gasi
# ujemnych tak ostro jak ReLU; GELU wazona probabilistycznie, standard w transformerach.
# Wyjscie: na zyczenie SiLU/GELU (OUTPUT_ACT), ALE sa nieograniczone (SiLU->(-0.28,+inf)),
# wiec predykcje klampujemy do [0,1] przy metrykach/rankingu. Jesli chcesz scisle 0-1
# z sieci, ustaw OUTPUT_ACT='none' + sigmoid — odkomentuj wariant ponizej.
ACTS = {"silu": nn.SiLU(), "gelu": nn.GELU(), "relu": nn.ReLU()}

class ExclusionMLP(nn.Module):
    def __init__(self, in_dim, hidden=HIDDEN, hidden_act=HIDDEN_ACT, output_act=OUTPUT_ACT, dropout=DROPOUT):
        super().__init__()
        h, layers, prev = list(hidden), [], in_dim
        for w in h:
            layers += [nn.Linear(prev, w), ACTS[hidden_act], nn.Dropout(dropout)]
            prev = w
        layers += [nn.Linear(prev, 1)]
        self.net = nn.Sequential(*layers)
        self.out = {"silu": nn.SiLU(), "gelu": nn.GELU(), "none": nn.Identity()}[output_act]
        # wariant scisle 0-1: self.out = nn.Sigmoid()
    def forward(self, x):
        return self.out(self.net(x)).squeeze(-1)

model = ExclusionMLP(X.shape[1]).to(device)
print(model)
print("parametry:", sum(p.numel() for p in model.parameters()))

In [ ]:
# ===== TRENING: Adam + MSE + early stopping =====
def to_t(a): return torch.tensor(a, dtype=torch.float32, device=device)
Xtr, Xva, Xte = map(to_t, Xs)
ytr, yva, yte = map(to_t, (y[tr_idx], y[va_idx], y[te_idx]))
tr_loader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(Xtr, ytr), batch_size=BATCH, shuffle=True)
opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=15)
lossf = nn.MSELoss()
hist, best, bad, best_state = {"tr": [], "va": []}, float("inf"), 0, None
for ep in range(1, EPOCHS + 1):
    model.train()
    tl = 0.0
    for xb, yb in tr_loader:
        opt.zero_grad(); loss = lossf(model(xb), yb); loss.backward(); opt.step()
        tl += loss.item() * len(xb)
    tl /= len(Xtr)
    model.eval()
    with torch.no_grad():
        vl = lossf(model(Xva), yva).item()
    sched.step(vl); hist["tr"].append(tl); hist["va"].append(vl)
    if vl < best - 1e-5: best, bad, best_state = vl, 0, {k: v.cpu().clone() for k, v in model.state_dict().items()}
    else: bad += 1
    if ep % 25 == 0 or bad == 0: print(f"ep {ep:3d} train={tl:.4f} val={vl:.4f} best={best:.4f}")
    if bad >= PATIENCE: print(f"early stop @ {ep}"); break
model.load_state_dict(best_state)
plt.plot(hist["tr"], label="train MSE"); plt.plot(hist["va"], label="val MSE")
plt.legend(); plt.xlabel("epoka"); plt.grid(True); plt.show()

In [ ]:
# ===== EWALUACJA + RANKING =====
model.eval()
with torch.no_grad():
    pred_all = model(to_t(sc.transform(X))).cpu().numpy()
pred_all = np.clip(pred_all, 0, 1)  # SiLU/GELU na wyjsciu -> dociecie do zakresu etykiety
te_pred = pred_all[te_idx]
print(f"TEST MAE={mean_absolute_error(y[te_idx], te_pred):.4f} RMSE={math.sqrt(mean_squared_error(y[te_idx], te_pred)):.4f} R2={r2_score(y[te_idx], te_pred):.4f} Spearman={spearmanr(y[te_idx], te_pred).statistic:.4f}")
for k in (20, 50):
    t, p = set(np.argsort(-y[te_idx])[:k]), set(np.argsort(-te_pred)[:k])
    print(f"Precision@{k} (test) = {len(t & p) / k:.2f}")
plt.scatter(y[te_idx], te_pred, s=8); plt.plot([0, 1], [0, 1], "r--")
plt.xlabel("etykieta"); plt.ylabel("predykcja"); plt.grid(True); plt.show()

In [ ]:
# ===== OUTPUT: lista heksagonow =====
out = df[["h3", "dzielnica", "lon", "lat", "etykieta_target"]].copy()
out["pred_wykluczenie"] = np.round(pred_all, 6)
out = out.sort_values("pred_wykluczenie", ascending=False).reset_index(drop=True)
out["rank"] = out.index + 1
out.to_csv("/content/lista_heksagonow_ranking.csv", index=False)
torch.save(model.state_dict(), "/content/model_silu_gelu.pt")
print(out.head(20).to_string(index=False))
print("zapisano: /content/lista_heksagonow_ranking.csv, /content/model_silu_gelu.pt")

## Hyperparameter grid (joint: definicja etykiety × sieć)

Uruchom **wszystkie komórki powyżej** przed gridem (model, dane, `device`).
Poziom 1: `ALPHA × W_TRAM × W_METRO` (`W_BUS=1.0` jako jednostka) — każda kombinacja przelicza etykietę.
Poziom 2: `HIDDEN × ACT × LR` przy stałym `dropout/wd`.
Percentyle p1/p99 etykiety liczone na **całym zbiorze** = stała definicja (udokumentowane uproszczenie).
Selekcja po **val Spearman** (ranking heksagonów), nie RMSE — skale targetu różnią się między definicjami.
Pełny faktorial po odcieciu par bezsensownych (w_tram < w_metro): 90 definicji × 8 konfiguracji = **720 treningów**.

In [ ]:
# ===== GRID: przestrzen =====
import itertools
GRID_LABEL = {"ALPHA": [0.2, 0.3, 0.5], "W_TRAM": [1.0, 1.5, 2.0, 3.0], "W_METRO": [1.5, 2.0, 3.0, 5.0], "W_KOLEJ": [1.5, 2.0, 3.0], "W_BUS": 1.0}
GRID_NET = {"HIDDEN": [[32, 16], [64, 32]], "ACT": ["silu", "gelu"], "LR": [1e-3, 3e-4]}
GRID_DROPOUT, GRID_WD, GRID_SEED = 0.3, 1e-4, 42
GRID_TOPK = 5  # ilu zwycięzców screening do uscislenia
n_label = len(GRID_LABEL["ALPHA"]) * len(GRID_LABEL["W_TRAM"]) * len(GRID_LABEL["W_METRO"]) * len(GRID_LABEL["W_KOLEJ"])
n_net = len(GRID_NET["HIDDEN"]) * len(GRID_NET["ACT"]) * len(GRID_NET["LR"])
print(f"definicje etykiety: {n_label}, konfigi sieci: {n_net}, razem: {n_label * n_net}")


In [ ]:
# ===== GRID: funkcje (etykieta / cechy / trening) =====
def grid_label(frame, alpha, w_tram, w_bus, w_metro, w_kolej):
    lp = np.log1p(frame["gestosc_os_km2"].clip(lower=0).to_numpy(float))
    lj = np.log1p(frame["pracownicy_szac"].clip(lower=0).to_numpy(float))
    z = (w_tram * frame["przystanki_tramwajowe"].to_numpy(float)
         + w_bus * frame["przystanki_autobusowe"].to_numpy(float)
         + w_metro * (frame["czy_metro"].to_numpy(float) >= 0.5).astype(float) + w_kolej * (frame.get("czy_kolej", 0).to_numpy(float) >= 0.5).astype(float))
    lz = np.log1p(np.clip(z, 0, None))
    def _mm(v):
        lo, hi = v.min(), v.max()
        return np.zeros_like(v) if hi <= lo else (v - lo) / (hi - lo)
    D = frame["wspolczynnik przyrostu"].to_numpy(float).clip(0, 1)
    raw = (alpha + alpha * D) * (_mm(lp) + _mm(lj)) - _mm(lz)
    p1, p99 = np.percentile(raw, [1, 99])
    lab = np.clip((raw - p1) / (p99 - p1), 0, 1) if p99 > p1 else np.zeros_like(raw)
    return lab

def grid_X(frame, w_tram, w_bus, w_metro, w_kolej, use_dz=USE_DZIELNICA):
    g = np.log1p(frame["gestosc_os_km2"].clip(lower=0).to_numpy(float))
    p = np.log1p(frame["pracownicy_szac"].clip(lower=0).to_numpy(float))
    z = (w_tram * frame["przystanki_tramwajowe"].to_numpy(float)
         + w_bus * frame["przystanki_autobusowe"].to_numpy(float)
         + w_metro * (frame["czy_metro"].to_numpy(float) >= 0.5).astype(float) + w_kolej * (frame.get("czy_kolej", 0).to_numpy(float) >= 0.5).astype(float))
    base = np.column_stack([g, p, np.log1p(np.clip(z, 0, None)),
                            frame["wspolczynnik przyrostu"].to_numpy(float),
                            frame["lon"].to_numpy(float), frame["lat"].to_numpy(float)])
    if use_dz:
        return np.hstack([base, pd.get_dummies(frame["dzielnica"].fillna("UNK"), prefix="dz").to_numpy(float)])
    return base

def grid_split(Xm, ym, groups, seed=GRID_SEED):
    gss = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=seed)
    tr, te = next(gss.split(Xm, ym, groups))
    gss2 = GroupShuffleSplit(n_splits=1, test_size=0.176, random_state=seed)
    a, b = next(gss2.split(Xm[tr], ym[tr], groups[tr]))
    return tr[a], tr[b], te

def grid_train(Xtr, ytr, Xva, yva, hidden, act, lr, wd, seed, epochs=EPOCHS, patience=PATIENCE):
    torch.manual_seed(seed); np.random.seed(seed)
    m = ExclusionMLP(Xtr.shape[1], hidden=hidden, hidden_act=act, output_act=act,
                     dropout=GRID_DROPOUT).to(device)
    dl = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(torch.tensor(Xtr, dtype=torch.float32),
                                       torch.tensor(ytr, dtype=torch.float32)),
        batch_size=BATCH, shuffle=True)
    opt = torch.optim.Adam(m.parameters(), lr=lr, weight_decay=wd)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=15)
    lf, best, bad, bs = nn.MSELoss(), float("inf"), 0, None
    Xt = torch.tensor(Xtr, dtype=torch.float32, device=device)
    Xv = torch.tensor(Xva, dtype=torch.float32, device=device)
    Yv = torch.tensor(yva, dtype=torch.float32, device=device)
    for _ in range(epochs):
        m.train()
        for xb, yb in dl:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad(); l = lf(m(xb), yb); l.backward(); opt.step()
        m.eval()
        with torch.no_grad():
            vl = lf(m(Xv), Yv).item()
        sched.step(vl)
        if vl < best - 1e-5: best, bad, bs = vl, 0, {k: v.cpu().clone() for k, v in m.state_dict().items()}
        else: bad += 1
        if bad >= patience: break
    m.load_state_dict(bs)
    m.eval()
    with torch.no_grad():
        pv = m(Xv).cpu().numpy()
    pv = np.clip(pv, 0, 1)
    return math.sqrt(mean_squared_error(yva, pv)), spearmanr(yva, pv).statistic, m


In [ ]:
# ===== GRID: screening 720 treningow (90 definicji x 8, 1 seed, staly split) =====
import time
groups_all = df["dzielnica"].fillna("UNK").to_numpy()
rows, t0 = [], time.time()
label_combos = [c for c in itertools.product(GRID_LABEL["ALPHA"], GRID_LABEL["W_TRAM"], GRID_LABEL["W_METRO"], GRID_LABEL["W_KOLEJ"]) if c[1] < c[2]]  # hierarchia: tramwaj < metro
net_combos = list(itertools.product(GRID_NET["HIDDEN"], GRID_NET["ACT"], GRID_NET["LR"]))
for ai, (alpha, wt, wm, wk) in enumerate(label_combos):
    y_g = grid_label(df, alpha, wt, GRID_LABEL["W_BUS"], wm, wk)
    X_g = grid_X(df, wt, GRID_LABEL["W_BUS"], wm, wk)
    tri, vai, tei = grid_split(X_g, y_g, groups_all)
    sc_g = StandardScaler().fit(X_g[tri])
    Xs = (sc_g.transform(X_g[tri]), sc_g.transform(X_g[vai]), sc_g.transform(X_g[tei]))
    ys = (y_g[tri], y_g[vai], y_g[tei])
    for hidden, act, lr in net_combos:
        rmse, sp, _ = grid_train(Xs[0], ys[0], Xs[1], ys[1], hidden, act, lr, GRID_WD, GRID_SEED)
        rows.append({"alpha": alpha, "w_tram": wt, "w_metro": wm, "w_kolej": wk, "hidden": str(hidden),
                     "act": act, "lr": lr, "val_rmse": round(rmse, 5), "val_spearman": round(float(sp), 5)})
    print(f"[{ai+1}/{len(label_combos)}] alpha={alpha} w_tram={wt} w_metro={wm} w_kolej={wk} — {len(rows)} treningow, {(time.time()-t0)/60:.1f} min")
res = pd.DataFrame(rows).sort_values("val_spearman", ascending=False).reset_index(drop=True)
res.to_csv("/content/grid_screening.csv", index=False)
print(res.head(10).to_string(index=False))
print(f"czas: {(time.time()-t0)/60:.1f} min | zapisano /content/grid_screening.csv")


In [ ]:
# ===== GRID: uscislanie top-5 (GroupKFold x 3 seedy) =====
from sklearn.model_selection import GroupKFold
REF_SEEDS = [42, 43, 44]
ref_rows = []
for k in range(min(GRID_TOPK, len(res))):
    r = res.iloc[k]
    y_g = grid_label(df, r["alpha"], r["w_tram"], GRID_LABEL["W_BUS"], r["w_metro"], r["w_kolej"])
    X_g = grid_X(df, r["w_tram"], GRID_LABEL["W_BUS"], r["w_metro"], r["w_kolej"])
    hidden = [int(x) for x in str(r["hidden"]).strip("[]").split(",")]
    sps = []
    for sd in REF_SEEDS:
        kf = GroupKFold(n_splits=5)
        for tri, vai in kf.split(X_g, y_g, groups_all):
            sc_g = StandardScaler().fit(X_g[tri])
            _, sp, _ = grid_train(sc_g.transform(X_g[tri]), y_g[tri], sc_g.transform(X_g[vai]),
                                  y_g[vai], hidden, r["act"], r["lr"], GRID_WD, sd)
            sps.append(float(sp))
    ref_rows.append({"alpha": r["alpha"], "w_tram": r["w_tram"], "w_metro": r["w_metro"], "w_kolej": r["w_kolej"],
                     "hidden": r["hidden"], "act": r["act"], "lr": r["lr"],
                     "cv_spear_mean": round(float(np.mean(sps)), 4),
                     "cv_spear_std": round(float(np.std(sps)), 4)})
    print(f"top{k+1}: mean={np.mean(sps):.4f} std={np.std(sps):.4f}")
ref = pd.DataFrame(ref_rows).sort_values("cv_spear_mean", ascending=False).reset_index(drop=True)
ref.to_csv("/content/grid_refinement.csv", index=False)
print(ref.to_string(index=False))
WIN = ref.iloc[0].to_dict()
print("ZWYCIEZCA:", WIN)


In [ ]:
# ===== FINAL: re-trening zwyciezcy na train+val, test raz, ranking =====
w_alpha, w_tram, w_metro, w_kolej = WIN["alpha"], WIN["w_tram"], WIN["w_metro"], WIN["w_kolej"]
w_hidden = [int(x) for x in str(WIN["hidden"]).strip("[]").split(",")]
y_f = grid_label(df, w_alpha, w_tram, GRID_LABEL["W_BUS"], w_metro, w_kolej)
X_f = grid_X(df, w_tram, GRID_LABEL["W_BUS"], w_metro, w_kolej)
tri, vai, tei = grid_split(X_f, y_f, groups_all, seed=GRID_SEED)
sc_f = StandardScaler().fit(X_f[np.concatenate([tri, vai])])
Xtrv = sc_f.transform(X_f[np.concatenate([tri, vai])])
ytrv = y_f[np.concatenate([tri, vai])]
Xte_f = sc_f.transform(X_f[tei])
torch.manual_seed(GRID_SEED); np.random.seed(GRID_SEED)
final = ExclusionMLP(X_f.shape[1], hidden=w_hidden, hidden_act=WIN["act"],
                     output_act=WIN["act"], dropout=GRID_DROPOUT).to(device)
dlf = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(
    torch.tensor(Xtrv, dtype=torch.float32), torch.tensor(ytrv, dtype=torch.float32)),
    batch_size=BATCH, shuffle=True)
optf = torch.optim.Adam(final.parameters(), lr=WIN["lr"], weight_decay=GRID_WD)
lf = nn.MSELoss()
final.train()
for _ in range(EPOCHS):
    for xb, yb in dlf:
        xb, yb = xb.to(device), yb.to(device)
        optf.zero_grad(); l = lf(final(xb), yb); l.backward(); optf.step()
final.eval()
with torch.no_grad():
    pall = np.clip(final(torch.tensor(sc_f.transform(X_f), dtype=torch.float32, device=device)).cpu().numpy(), 0, 1)
pte = pall[tei]
print(f"TEST (etykieta zwyciezcy): MAE={mean_absolute_error(y_f[tei], pte):.4f} "
      f"RMSE={math.sqrt(mean_squared_error(y_f[tei], pte)):.4f} R2={r2_score(y_f[tei], pte):.4f} "
      f"Spearman={spearmanr(y_f[tei], pte).statistic:.4f}")
out = df[["h3", "dzielnica", "lon", "lat"]].copy()
out["etykieta_final"] = np.round(y_f, 6)
out["pred_wykluczenie"] = np.round(pall, 6)
out = out.sort_values("pred_wykluczenie", ascending=False).reset_index(drop=True)
out["rank"] = out.index + 1
out.to_csv("/content/lista_heksagonow_ranking_best.csv", index=False)
TOP_K = 30  # liczba heksagonow o najwyzszym wskazniku do osobnego pliku
out.head(TOP_K).to_csv("/content/lista_heksagonow_top30.csv", index=False)
torch.save({"state": final.state_dict(), "winner": WIN, "scaler_mean": sc_f.mean_, "scaler_scale": sc_f.scale_},
           "/content/model_best.pt")
print(out.head(20).to_string(index=False))
print("zapisano: /content/lista_heksagonow_ranking_best.csv, /content/lista_heksagonow_top30.csv, /content/model_best.pt")


## Heatmapa heksagonów (folium)

Wymaga uruchomienia komórki finałowej (`out` z rankingiem). Kolor = `pred_wykluczenie`, czarny obrys = top-30.
Zapis: `/content/heatmapa_wykluczenia.html` (do pobrania z panelu Files).

In [ ]:
# ===== HEATMAPA: join geometrii z rankingiem =====
import folium, json
from branca.colormap import LinearColormap
try:
    GEO_PATH  # z komorki pobierania; jesli brak -> odtworz sciezke
except NameError:
    GEO_PATH = os.path.join(LOCAL_DIR, "warszawa_h3_400m_pelna.geojson")
try:
    TOP_K
except NameError:
    TOP_K = 30
rank_df = out[["h3", "dzielnica", "pred_wykluczenie", "rank"]].copy()
with open(GEO_PATH, encoding="utf-8") as f:
    geo = json.load(f)
pred = dict(zip(rank_df["h3"], rank_df["pred_wykluczenie"]))
rnk = dict(zip(rank_df["h3"], rank_df["rank"]))
matched, missing = 0, []
for feat in geo["features"]:
    h = feat["properties"].get("h3")
    if h in pred:
        feat["properties"]["pred"] = round(float(pred[h]), 4)
        feat["properties"]["rank"] = int(rnk[h])
        feat["properties"]["status"] = "analiza"
        matched += 1
    else:
        feat["properties"]["pred"] = -1.0
        feat["properties"]["rank"] = -1
        feat["properties"]["status"] = "poza analiza (clean)"
        missing.append(h)
print(f"dopasowano {matched}/{len(geo['features'])} geometrii" + (f", brak: {missing[:5]}" if missing else ""))
cmap = LinearColormap(["#ffffcc", "#fed976", "#feb24c", "#fd8d3c", "#e31a1c", "#800026"],
                      vmin=0, vmax=1, caption="pred_wykluczenie")
top30 = set(rank_df[rank_df["rank"] <= TOP_K]["h3"])
m = folium.Map(location=[float(out["lat"].mean()), float(out["lon"].mean())],
               zoom_start=11, tiles="CartoDB positron")
folium.GeoJson(
    geo, name="wskaznik wykluczenia",
    style_function=lambda f: ({"fillColor": "#cccccc", "color": "#888888", "weight": 0.3, "fillOpacity": 0.4}
                              if f["properties"].get("status") != "analiza" else
                              {"fillColor": cmap(f["properties"]["pred"]),
                              "color": "black", "weight": 0.3, "fillOpacity": 0.7}),
    tooltip=folium.GeoJsonTooltip(fields=["h3", "dzielnica", "pred", "rank", "status"],
                                  aliases=["h3", "dzielnica", "pred", "rank", "status"]),
).add_to(m)
folium.GeoJson(
    {"type": "FeatureCollection",
     "features": [f for f in geo["features"] if f["properties"].get("h3") in top30]},
    name=f"top-{TOP_K}", style_function=lambda f: {"color": "black", "weight": 2.5, "fillOpacity": 0},
).add_to(m)
cmap.add_to(m); folium.LayerControl().add_to(m)
try:
    _banner = f"(alpha={w_alpha} tram={w_tram} metro={w_metro} kolej={w_kolej})"
except NameError:
    _banner = f"(alpha={ALPHA} tram={W_TRAM} metro={W_METRO} kolej={W_KOLEJ})"
m.get_root().html.add_child(folium.Element(
    f"<div style='position:fixed;top:10px;left:50px;z-index:999;background:white;padding:6px 10px;"
    f"border:1px solid grey;font-size:13px'>Wskaznik wykluczenia — top-{TOP_K} obrysowane "
    f"{_banner}</div>"))
m.save("/content/heatmapa_wykluczenia.html")
print("zapisano: /content/heatmapa_wykluczenia.html")
for feat in geo["features"]:  # flaga top-K + zapis jako GeoJSON
    feat["properties"]["top30"] = 1 if feat["properties"].get("h3") in top30 else 0
with open("/content/heatmapa_wykluczenia.geojson", "w", encoding="utf-8") as f:
    json.dump(geo, f, ensure_ascii=False)
print("zapisano: /content/heatmapa_wykluczenia.geojson")
m
